In [1]:
from pathlib import Path
import pandas as pd

data_dir = Path("../data/raw/neso_demand")

files = sorted(data_dir.glob("*.csv"))

print("Files found:")
for file in files:
    print("-", file.name)

print(f"\nNumber of files: {len(files)}")

Files found:
- neso_demand_2020.csv

Number of files: 1


In [2]:
file_path = files[0]

df = pd.read_csv(file_path)

print(f"Rows: {len(df):,}")
print(f"Columns: {len(df.columns)}")

print("\nColumns:")
print(df.columns.tolist())

Rows: 17,568
Columns: 22

Columns:
['_id', 'SETTLEMENT_DATE', 'SETTLEMENT_PERIOD', 'ND', 'TSD', 'ENGLAND_WALES_DEMAND', 'EMBEDDED_WIND_GENERATION', 'EMBEDDED_WIND_CAPACITY', 'EMBEDDED_SOLAR_GENERATION', 'EMBEDDED_SOLAR_CAPACITY', 'NON_BM_STOR', 'PUMP_STORAGE_PUMPING', 'IFA_FLOW', 'IFA2_FLOW', 'BRITNED_FLOW', 'MOYLE_FLOW', 'EAST_WEST_FLOW', 'NEMO_FLOW', 'NSL_FLOW', 'ELECLINK_FLOW', 'VIKING_FLOW', 'GREENLINK_FLOW']


In [3]:
df["SETTLEMENT_DATE"] = pd.to_datetime(
    df["SETTLEMENT_DATE"],
    dayfirst=True
)

print("Minimum date:", df["SETTLEMENT_DATE"].min())
print("Maximum date:", df["SETTLEMENT_DATE"].max())
print(
    "Unique dates:",
    df["SETTLEMENT_DATE"].nunique()
)

Minimum date: 2020-01-01 00:00:00
Maximum date: 2020-12-31 00:00:00
Unique dates: 366


C:\Users\USER\AppData\Local\Temp\ipykernel_6364\2577562759.py:1: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df["SETTLEMENT_DATE"] = pd.to_datetime(


In [4]:
period_counts = (
    df.groupby("SETTLEMENT_DATE")["SETTLEMENT_PERIOD"]
    .nunique()
)

print(
    period_counts.value_counts()
    .sort_index()
)

SETTLEMENT_PERIOD
46      1
48    364
50      1
Name: count, dtype: int64


In [5]:
print(df.isna().sum())

_id                          0
SETTLEMENT_DATE              0
SETTLEMENT_PERIOD            0
ND                           0
TSD                          0
ENGLAND_WALES_DEMAND         0
EMBEDDED_WIND_GENERATION     0
EMBEDDED_WIND_CAPACITY       0
EMBEDDED_SOLAR_GENERATION    0
EMBEDDED_SOLAR_CAPACITY      0
NON_BM_STOR                  0
PUMP_STORAGE_PUMPING         0
IFA_FLOW                     0
IFA2_FLOW                    0
BRITNED_FLOW                 0
MOYLE_FLOW                   0
EAST_WEST_FLOW               0
NEMO_FLOW                    0
NSL_FLOW                     0
ELECLINK_FLOW                0
VIKING_FLOW                  0
GREENLINK_FLOW               0
dtype: int64


In [6]:
duplicate_count = df.duplicated(
    subset=[
        "SETTLEMENT_DATE",
        "SETTLEMENT_PERIOD"
    ]
).sum()

print(
    f"Duplicate settlement observations: "
    f"{duplicate_count:,}"
)

Duplicate settlement observations: 0


In [7]:
print(
    df["ND"].describe()
)

count    17568.000000
mean     27156.527607
std       6645.952421
min      13367.000000
25%      21890.500000
50%      26053.000000
75%      31749.250000
max      45986.000000
Name: ND, dtype: float64


In [8]:
print(
    "Negative ND values:",
    (df["ND"] < 0).sum()
)

Negative ND values: 0


In [9]:
expected = 366 * 48
actual = len(df)

print(f"Expected: {expected:,}")
print(f"Actual:   {actual:,}")
print(f"Difference: {actual - expected:,}")

Expected: 17,568
Actual:   17,568
Difference: 0


In [ ]:
from pathlib import Path
import pandas as pd

data_dir = Path("../data/raw/neso_demand")
files = sorted(data_dir.glob("*.csv"))

summary = []

for file in files:
    df_year = pd.read_csv(file)

    df_year["SETTLEMENT_DATE"] = pd.to_datetime(
        df_year["SETTLEMENT_DATE"],
        format="%d-%b-%Y"
    )

    summary.append({
        "year": file.stem[-4:],
        "rows": len(df_year),
        "min_date": df_year["SETTLEMENT_DATE"].min(),
        "max_date": df_year["SETTLEMENT_DATE"].max(),
        "unique_dates": df_year["SETTLEMENT_DATE"].nunique(),
        "duplicates": df_year.duplicated(
            subset=[
                "SETTLEMENT_DATE",
                "SETTLEMENT_PERIOD"
            ]
        ).sum(),
        "missing_values": df_year[
            ["SETTLEMENT_DATE", "SETTLEMENT_PERIOD", "ND"]
        ].isna().sum().sum(),
    })

summary_df = pd.DataFrame(summary)

summary_df

C:\Users\USER\AppData\Local\Temp\ipykernel_6364\605265480.py:12: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df_year["SETTLEMENT_DATE"] = pd.to_datetime(
C:\Users\USER\AppData\Local\Temp\ipykernel_6364\605265480.py:12: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df_year["SETTLEMENT_DATE"] = pd.to_datetime(
C:\Users\USER\AppData\Local\Temp\ipykernel_6364\605265480.py:12: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df_year["SETTLEMENT_DATE"] = pd.to_datetime(
C:\Users\USER\AppData\Local\Temp\ipykernel_6364\605265480.py:12: UserWarning: Could not infer format, so each el

ValueError: unconverted data remains when parsing with format "%Y-%d-%m": "3". You might want to try:
    - passing `format` if your strings have a consistent format;
    - passing `format='ISO8601'` if your strings are all ISO8601 but not necessarily in exactly the same format;
    - passing `format='mixed'`, and the format will be inferred for each element individually. You might want to use `dayfirst` alongside this.